# Accuracy, entropy and variation between clustering runs

[![Open in Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/gromicho/teaching/blob/main/courses/abw/notebooks/lecture-4/accuracy-entropy-and-clustering.ipynb) [![Open in Binder](https://mybinder.org/badge_logo.svg)](https://mybinder.org/v2/gh/gromicho/teaching/main?urlpath=tree/courses/abw/notebooks/lecture-4/accuracy-entropy-and-clustering.ipynb)

ABW lecture companion, originally developed by the ABW teaching team. This edition preserves the original sequence of data inspection, models and interpretation. Predict each result before running the cell.


## Prepare the libraries and original teaching data
Install missing libraries, then load the verified raw fruit table. Corrections happen in the lesson below, after inspecting the observations.


In [ ]:
# Load the shared teaching utilities from this checkout or a verified download.
from pathlib import Path
import hashlib
import sys
from urllib.request import urlopen

support_path = next((folder / 'support' for folder in [Path.cwd(), *Path.cwd().parents]
                     if (folder / 'support' / 'teaching_utils.py').is_file()), None)
if support_path is None:
    support_path = Path.cwd() / '.teaching-support'
    support_path.mkdir(exist_ok=True)
    helper = support_path / 'teaching_utils.py'
    expected = 'fbfa41e12709a01548e213abba976dad1e21d0abcfd798a2dfd669706cc152bb'
    if not helper.exists() or hashlib.sha256(helper.read_bytes()).hexdigest() != expected:
        url = 'https://raw.githubusercontent.com/gromicho/teaching/f3ad11b77cae7dd05315d314c7e72ee8516aaa3d/support/teaching_utils.py'
        content = urlopen(url, timeout=45).read()
        if hashlib.sha256(content).hexdigest() != expected:
            raise ValueError('Teaching helper version changed; reopen the current course notebook.')
        helper.write_bytes(content)
sys.path.insert(0, str(support_path))
from teaching_utils import ensure_packages

required_packages = {'matplotlib': 'matplotlib', 'pandas': 'pandas', 'numpy': 'numpy', 'seaborn': 'seaborn', 'scipy': 'scipy', 'sklearn': 'scikit-learn'}
ensure_packages(required_packages)


In [ ]:
from pathlib import Path
from urllib.request import urlopen
import hashlib

# Prefer the checked-in file locally; Colab downloads the same frozen edition.
data_path = next((p for p in [Path("data/fruits.csv"), Path("fruits.csv")]
                 if p.is_file()), Path("fruits.csv"))
if not data_path.is_file():
    url = "https://raw.githubusercontent.com/gromicho/teaching/main/data/fruits.csv"
    with urlopen(url, timeout=45) as response:
        payload = response.read()
    if hashlib.sha256(payload).hexdigest() != "84235de12e6a8eb094423436b3cdd045c8b55288d773053c821d88db06443c22":
        raise ValueError("Dataset checksum mismatch; do not use an unverified copy.")
    data_path.write_bytes(payload)
assert hashlib.sha256(data_path.read_bytes()).hexdigest() == "84235de12e6a8eb094423436b3cdd045c8b55288d773053c821d88db06443c22", "Unexpected local data version"


In [ ]:
import pandas as pd
import seaborn as sns
import numpy as np
import matplotlib.pyplot as plt
from sklearn import tree
from sklearn.cluster import KMeans
from sklearn.linear_model import LinearRegression
from itertools import combinations

In [ ]:
the_markers = {
'1': 'Tri_down marker',
'2': 'Tri_up marker',
'3': 'Tri_left marker',
'4': 'Tri_right marker',
'.': 'Point marker',
',': 'Pixel marker',
'o': 'Circle marker',
'v': 'Triangle_down marker',
'^': 'Triangle_up marker',
'<': 'Triangle_left marker',
'>': 'Triangle_right marker',
's': 'Square marker',
'p': 'Pentagon marker',
'*': 'Star marker',
'h': 'Hexagon1 marker',
'H': 'Hexagon2 marker',
'+': 'Plus marker',
'x': 'X marker',
'D': 'Diamond marker',
'd': 'Thin_diamond marker'}

the_colors = {
'r': 'Red',
'g': 'Green',
'b': 'Blue',
'm': 'Magenta',
'y': 'Yellow',
'c': 'Cyan',
'k': 'Black',
'w': 'White'
}

colors = list(the_colors.keys())
markers = list(the_markers.keys())

In [ ]:
def plot_classifier(ax, estimator, X, y,
                   steps=500, colors=colors, markers=markers, ticks=True):
    if X.shape[1] != 2:
        raise ValueError("X must be 2D")

    targets = np.unique(y)
    categories = { c:i for i,c in enumerate(targets) }
    nof_categories = len(categories)

    x_low = np.min(X[:, 0])
    x_high = np.max(X[:, 0])
    y_low = np.min(X[:, 1])
    y_high = np.max(X[:, 1])

    x_extra = (x_high - x_low) * 0.1
    y_extra = (y_high - y_low) * 0.1

    x_low -= x_extra
    x_high += x_extra
    y_low -= y_extra
    y_high += y_extra

    xx, yy = np.meshgrid(
        np.linspace(x_low, x_high, steps), np.linspace(y_low, y_high, steps)
    )

    vectorized_index = np.vectorize(lambda x : categories[x])

    Z = estimator.predict(np.c_[xx.ravel(), yy.ravel()])
    Z = vectorized_index( Z.reshape(xx.shape) )

    ax.contourf(xx, yy, Z, alpha=0.25,
                levels=[level-.1 for level in range(nof_categories+1)],
                colors=colors)

    for i,t in enumerate(targets):
        c = colors[i]
        m = markers[i]
        idx = y == t
        ax.scatter(X[idx,0], X[idx,1], marker=m, c=c, label=t)

    ax.set_xlim(x_low, x_high)
    ax.set_ylim(y_low, y_high)

    ax.get_xaxis().set_visible(ticks)
    ax.get_yaxis().set_visible(ticks)

    ax.legend()

    return ax

# Get the fruit data

For the reasoning behind the function `FixTheFruitOutliers` see the previous lecture.

In [ ]:
# Reuse the correction explained step by step in Lecture 3.
helper = support_path / 'fruit_utils.py'
expected = '8ee86a20d049c601405ef29c24c3ac40a0f8a824ce9d697d285acdf41f8c45bd'
if not helper.exists():
    payload = urlopen('https://raw.githubusercontent.com/gromicho/teaching/0e13f045e004a70261e0a7f612a62f96ccb171b8/support/fruit_utils.py', timeout=45).read()
    if hashlib.sha256(payload).hexdigest() != expected:
        raise ValueError('Fruit helper checksum mismatch')
    helper.write_bytes(payload)
if hashlib.sha256(helper.read_bytes()).hexdigest() != expected:
    raise ValueError('Unexpected fruit helper version')
from fruit_utils import fix_fruit_outliers


In [ ]:
fruits = fix_fruit_outliers(pd.read_csv(data_path, delimiter=';', decimal=','))
fruits


# Visualize the data

In [ ]:
sns.scatterplot(x='Length', y='Width', data=fruits)
plt.show()

In [ ]:
sns.scatterplot(x='Length', y='Width', data=fruits, hue='Name')
plt.show()

# Classification trees
The errors counted below are **training errors**. A deeper tree can fit these observations better without predicting unseen observations better. Keep this distinction when discussing accuracy.


In [ ]:
def ShowTheseMaxDepths( max_depths,
                       X, y,
                       features,
                       criterion= 'entropy',
                       fontsize= 5,
                       steps= 200,
                       figsize= (13,5),
                       file_types=['pdf','svg']):
    for max_depth in max_depths:
        classifier = tree.DecisionTreeClassifier(
            criterion=criterion,
            max_depth=max_depth, random_state=0
        )
        classifier = classifier.fit( X, y )
        nof_misclassifications = sum(classifier.predict( X ) != y )
        fig, (ax_tree, ax_surface) = plt.subplots(1,2,figsize=figsize)
        _ = tree.plot_tree(classifier, ax=ax_tree,
                           filled=True, fontsize=fontsize, feature_names=features,
                           class_names=sorted( np.unique( y ) )
        )
        plot_classifier( ax_surface, classifier, X, y, steps=steps )
        fig.suptitle(f'{nof_misclassifications} misclassifications with depth {max_depth}')
        for ft in file_types:
            fig.savefig(f'max_{max_depth}_depth.{ft}', bbox_inches='tight')
        plt.show()

In [ ]:
fruits_features = ['Length','Width']
X = fruits[fruits_features].values
y = np.ravel( fruits.Name.values )

ShowTheseMaxDepths( range(10,15), X, y, fruits_features, file_types=[] )

In [ ]:
from sklearn.datasets import load_iris
iris_data = load_iris(as_frame=True)
iris = iris_data.data.copy()
iris.columns = ['sepal_length', 'sepal_width', 'petal_length', 'petal_width']
iris['species'] = [iris_data.target_names[i] for i in iris_data.target]
palette = 'Set1'
s = 65
fig, axs = plt.subplots( 2, 3, figsize=(13,8) )
for ax,(r,c) in zip(axs.flat, combinations(iris.columns[:4],2)):
    sns.scatterplot( ax=ax,
                    x=r, y=c,
                    data=iris, hue="species", palette=palette, s=s)

fig.suptitle("Iris Dataset")
handles, labels = axs.flat[0].get_legend_handles_labels()
fig.legend(handles, labels, loc='center right')
for ax in axs.flat[1:]:
    ax.legend().set_visible(False)
plt.show()

In [ ]:
iris_not_setosa = iris[iris.species != 'setosa']
features = ['petal_length', 'petal_width']
sns.scatterplot(
    x=features[0], y=features[1],
    data=iris_not_setosa,
    hue="species", palette="Set1", s=s
)
plt.show()

In [ ]:
X = iris_not_setosa[features].values
y = np.ravel( iris_not_setosa.species.values )

ShowTheseMaxDepths( range(2,5), X, y, features, file_types=[] )

In [ ]:
iris_not_versicolor = iris[iris.species != 'versicolor']
features = ['sepal_length', 'sepal_width']
sns.scatterplot(
    x=features[0], y=features[1],
    data=iris_not_versicolor,
    hue="species", palette="Set1", s=s
)
plt.show()

In [ ]:
X = iris_not_versicolor[features].values
y = np.ravel( iris_not_versicolor.species.values )

ShowTheseMaxDepths( range(2,4), X, y, features, file_types=[] )

# Added to explain better the concept of entropy

In [ ]:
from math import log2

def Entropy( p ):
  return sum( [ -p*log2(p) if p > 0 else 0 for p in p ])

def EntropyNormalized( p ):
  return Entropy( p ) / log2(len(p))

from scipy.stats import entropy

In [ ]:
p = [ .5, .5 ]
entropy(p,base=2),EntropyNormalized(p),Entropy(p)

In [ ]:
counts = fruits.Name.value_counts().to_dict()
count = counts.values()
sum(count)

In [ ]:
p = [ c/sum(count) for c in count]

In [ ]:
entropy(p,base=2),EntropyNormalized(p),Entropy(p)

In [ ]:
node = [0,20,25,2]
p = [ c/sum(node) for c in node]

In [ ]:
entropy(p,base=2),EntropyNormalized(p),Entropy(p)

In [ ]:
4/40*Entropy( [4/4, 0/4] )+36/40*Entropy( [16/36, 20/36] )

In [ ]:
Entropy( [16/36, 20/36] )

## Clustering and repeated runs
The six elbow plots deliberately use different seeds. A fixed sequence of distinct seeds makes the experiment reproducible while preserving variability between runs. `n_init=1` means one initialization per fitted model, so we can see the local optima rather than hiding the variation behind multiple restarts.


In [ ]:
data = fruits[fruits_features].copy()

In [ ]:
K = range(1, 11)
fig,axs = plt.subplots(2,3,figsize=(20,8))
for seed, ax in enumerate(axs.flat):
    inertias = []
    for k in K:
        kmeans = KMeans(n_clusters=k, n_init=1, random_state=seed)
        kmeans.fit(data)
        inertias.append(kmeans.inertia_)

    ax.plot(K, inertias, marker='o')
    ax.grid(True)
    ax.set_xlabel('Number of Clusters (K)')
    ax.set_ylabel('SSE')
    fig.suptitle('Elbow Method for Optimal K, different runs')
plt.show()

# Let us show that `kmeans` may find solutions of very different quality for the same k due to randomness

In [ ]:
k=7
inertias = dict()
for seed in range(1000):
    kmeans = KMeans(n_clusters=k, n_init=1, random_state=seed)
    kmeans.fit(data)
    inertias[kmeans.inertia_] = kmeans

values = sorted(inertias.keys())
best,worse = values[0],values[-1]


for i in [best,worse]:
    centroids = inertias[i].cluster_centers_
    fruits['cluster'] = inertias[i].labels_
    sns.scatterplot(x='Length', y='Width', data=fruits,palette=palette,hue='cluster')
    plt.scatter(centroids[:, 0], centroids[:, 1], c='red', marker='x', s=200, label='Centroids')
    plt.title(inertias[i].inertia_)
    plt.show()
assert worse > best, "Expected different local optima across the distinct starts"
print({"best_SSE": best, "worst_SSE": worse, "distinct_SSE_values": len(values)})


# Comparing the clusters with the actual fruits

In [ ]:
k = 3
kmeans = KMeans(n_clusters=k, n_init=10, random_state=0).fit(data)
fruits['cluster'] = kmeans.labels_
fig,(ax1,ax2) = plt.subplots(1,2,figsize=(13,5))
a,b=fruits_features
sns.scatterplot(ax=ax1,x=a,y=b,data=fruits,palette=palette,hue='cluster')
sns.scatterplot(ax=ax2,x=a,y=b,data=fruits,palette=palette,hue='Name')
plt.show()

# Linear regression

In [ ]:
lin_reg = LinearRegression()  # Create linear regression object
x_train = fruits[fruits.cluster==0].Length.values.reshape(-1,1)
y_train = fruits[fruits.cluster==0].Width.values.reshape(-1,1)
lin_reg.fit( x_train, y_train )  # Train the model using the training sets

In [ ]:
model_line = lin_reg.predict(x_train)
plt.scatter(x_train, y_train, color='black')
plt.plot(x_train, model_line, color='blue', linewidth=3)
plt.xticks(())
plt.yticks(())
plt.show()

In [ ]:
# Intercept: the value for y when x=0 for the predicted line, \beta_0 in the formulas
lin_reg.intercept_

In [ ]:
# Coefficient: the slope of the predicted line, \beta_1 in the formulas
lin_reg.coef_

In [ ]:
sns.lmplot(x=a,y=b,data=fruits,palette=palette,hue='cluster')
plt.show()

In [ ]:
import numpy as np
assert np.isclose(Entropy([0.5, 0.5]), 1)
assert np.isclose(Entropy([0, 1]), 0)
assert np.isclose(Entropy(p), entropy(p, base=2))
